# PHASE 1 - EEG PREPROCESSING

### STEP 1 - LOAD DATA 

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Imports
# ══════════════════════════════════════════════════════════════════════════════
from pathlib import Path          # voor bestandspaden (werkt op Windows en Mac)
import numpy as np                # numerieke berekeningen
import pandas as pd               # data opslaan als tabel
import mne                        # EDF bestanden inladen
from tqdm import tqdm 
from datetime import datetime
import re

# ══════════════════════════════════════════════════════════════════════════════
# Configuration - all data files are stored in these base directories + output folder
# ══════════════════════════════════════════════════════════════════════════════

BASE_DIRS = [
    Path(r"\\vs03.herseninstituut.knaw.nl\VS03-SandC-2\raw\bnbd\Data\eeg\NSR"),
    Path(r"\\vs03.herseninstituut.knaw.nl\VS03-SandC-2\raw\bnbd\Data\eeg\Prezens"),
    Path(r"\\vs03.herseninstituut.knaw.nl\VS03-SandC-2\raw\bnbd\Data\eeg\SAV"),
]
output_dir = Path(r"C:\Users\zafar\Documents\THESIS_OUTPUTS\1_preprocessing_EEG")
output_dir.mkdir(exist_ok=True)

EEG_CH = ['EEG L psg-lp', 'EEG R psg-lp']          # EEG channels left and right 
EMG_CH = ['EEG L psg-emg', 'EEG R psg-emg']        # EMG channels (muscle activity)
MOV_CH = ['dX', 'dY', 'dZ']                        # movement channels (accelerometer)
ALL_CH = EEG_CH + EMG_CH + MOV_CH                  # all channels together

L_FREQ   = 0.1                                     # Hz deletes slow drifts and DC offset 
H_FREQ   = 35.0                                    # Hz enough for all relevant EEG activity (delta, theta, alpha, beta)
NOTCH_HZ = 50.0 
SFREQ    = 256                                     # Original sampling frequency of the data
MOVEMENT_THRESHOLD_UV = 1000.0                     # Way above normal EEG (~200mV) 
TARGET_SFREQ = 128                                 # Resampling if needed
GROUPS = ["NSR", "Prezens", "SAV"]


In [35]:
# ══════════════════════════════════════════════════════════════════════════════
# Hulpfuncties
# ══════════════════════════════════════════════════════════════════════════════

def find_edf_files(base_dirs: list, limit: int = None) -> list:
    """
    Zoekt EDF-bestanden via rglob("*_psg.edf").
    Slaat de lijst op als edf_filelist.csv zodat het bij herstart niet opnieuw hoeft.
    Verwijder edf_filelist.csv handmatig als je een nieuwe scan wilt forceren.
    """
    cache_path = OUTPUT_DIR / "edf_filelist.csv"

    # ── Laad uit cache als die bestaat ───────────────────────────
    if cache_path.exists():
        cached = pd.read_csv(cache_path)["file_path"].tolist()
        all_files = [Path(p) for p in cached]
        print(f"  [CACHE] {len(all_files)} bestanden geladen uit {cache_path.name}")
        print(f"          (verwijder dit bestand om opnieuw te scannen)")

    # ── Anders: scan de mappen en sla op ─────────────────────────
    else:
        print("  [SCAN] Geen cache gevonden, mappen worden gescand...")
        all_files = []

        for base in base_dirs:
            if not base.exists():
                print(f"  [WARN] Niet bereikbaar: {base}")
                continue

            found = sorted(f for f in base.rglob("*_psg.edf") if "_T0_" in f.name)          # only T0 files       
            print(f"  {len(found):>5} gevonden (T0)  ← {base.name}")
            all_files.extend(found)

        # Opslaan als cache
        pd.DataFrame({"file_path": [str(f) for f in all_files]}).to_csv(cache_path, index=False)
        print(f"\n  [CACHE] Lijst opgeslagen: {cache_path.name}")

    if limit:
        print(f"  [INFO] Limiet: eerste {limit} bestanden")
        all_files = all_files[:limit]

    print(f"\n  → Totaal te verwerken: {len(all_files)} bestanden")
    return all_files


def extract_ids(edf_path: Path) -> dict:
    """
    Haalt subject_id en night_id uit de bestandsnaam.
    Exact zoals jouw originele code:

    Bestandsnaam: bnbd_nsr_03554_T0_N1_psg.edf
    stem:         bnbd_nsr_03554_T0_N1_psg
    na _psg:      bnbd_nsr_03554_T0_N1
    parts:        ['bnbd', 'nsr', '03554', 'T0', 'N1']

    subject_id →  bnbd_nsr_03554
    night_id   →  T0_N1
    group      →  nsr / sav / prezens (uit parts[1])
    """
    stem  = edf_path.stem                        # zonder .edf
    parts = stem.replace("_psg", "").split("_")  # verwijder _psg suffix

    subject_id = f"bnbd_{parts[1]}_{parts[2]}"  # bnbd_nsr_03554
    night_id   = f"{parts[3]}_{parts[4]}"        # T0_N1
    group      = parts[1].upper()                # NSR / SAV / PREZENS

    return {
        "subject_id":    subject_id,
        "night_id":      night_id,
        "group":         group,
        "participant_id": subject_id,            # alias voor compatibiliteit
    }

In [1]:
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import mne

OUTPUT_DIR = Path(r"C:\Users\zafar\Documents\THESIS_OUTPUTS\1_preprocessing_EEG")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ══════════════════════════════════════════════════════════════════════════════
# Preprocessing functions
# ══════════════════════════════════════════════════════════════════════════════

def load_night(edf_file: Path):
    """Laadt één EDF en past DC-removal, notch, bandpass en resampling toe."""
    raw = mne.io.read_raw_edf(edf_file, preload=False, verbose=False)

    available   = raw.ch_names
    ch_to_pick  = [ch for ch in ALL_CH if ch in available]
    eeg_present = [ch for ch in EEG_CH  if ch in available]
    emg_present = [ch for ch in EMG_CH  if ch in available]
    mov_present = [ch for ch in MOV_CH  if ch in available]
    missing     = [ch for ch in ALL_CH  if ch not in available]

    if missing:
        print(f"    [WARN] Kanalen niet gevonden: {missing}")
    if not eeg_present:
        raise ValueError("Geen EEG-kanalen aanwezig.")

    raw.pick(ch_to_pick)
    raw.load_data(verbose=False)
    raw._data = raw._data.astype(np.float64)

    sfreq_orig = raw.info["sfreq"]

    # 1. Delete DC offset 
    for ch in eeg_present:
        idx = raw.ch_names.index(ch)
        raw._data[idx] -= np.mean(raw._data[idx])

    # 2. Notch 50 Hz
    if sfreq_orig > NOTCH_HZ * 2:
        raw.notch_filter(freqs=NOTCH_HZ, picks=eeg_present, verbose=False)

    # 3. Bandpass EEG: 0.1–35 Hz
    raw.filter(l_freq=L_FREQ, h_freq=H_FREQ,
               picks=eeg_present, method="fir",
               fir_window="hamming", verbose=False)

    # 4. EMG filter
    if emg_present:
        h_emg = min(100.0, sfreq_orig / 2 - 1)
        raw.filter(l_freq=10.0, h_freq=h_emg, picks=emg_present, verbose=False)

    # 5. MOVEMENT: DC removal
    if mov_present:
        raw.apply_function(lambda x: x - np.mean(x), picks=mov_present, verbose=False)

    # 6. Resample naar 128 Hz
    if sfreq_orig != TARGET_SFREQ:
        raw.resample(TARGET_SFREQ, verbose=False)

    return raw


def preprocess_signals(raw) -> dict:
    """Geeft dictionary: kanaalnaam → 1D numpy array."""
    return {ch: raw.get_data(picks=ch)[0] for ch in ALL_CH if ch in raw.ch_names}


def remove_movement_artifacts(signals: dict, sfreq: int = TARGET_SFREQ) -> tuple:
    n_samples     = len(next(iter(signals.values())))
    combined_mask = np.zeros(n_samples, dtype=bool)
    buffer        = int(0.5 * sfreq)
    stats         = {}

    for ch in EEG_CH:
        if ch not in signals:
            continue

        # data is al in µV, dus direct vergelijken met drempel in µV
        mask         = np.abs(signals[ch]) > MOVEMENT_THRESHOLD_UV
        mask_dilated = np.zeros(n_samples, dtype=bool)
        for idx in np.where(mask)[0]:
            mask_dilated[max(0, idx - buffer):min(n_samples, idx + buffer)] = True
        stats[ch]     = round(mask_dilated.mean() * 100, 2)
        combined_mask |= mask_dilated

    return combined_mask, stats


# ══════════════════════════════════════════════════════════════════════════════
# Verwerking van één nacht
# ══════════════════════════════════════════════════════════════════════════════

def process_one_night(edf_path: Path) -> dict:
    """Volledige pipeline voor één EDF-bestand."""
    ids = extract_ids(edf_path)

    log = {
        "file_path":          str(edf_path),
        "group":              ids["group"],
        "subject_id":         ids["subject_id"],
        "night_id":           ids["night_id"],
        "status":             "failed",
        "error":              "",
        "sfreq_original":     "",
        "duration_hours":     "",
        "movement_pct_L":     "",
        "movement_pct_R":     "",
        "preprocessed_path":  "",
        "timestamp":          datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }

    try:
        raw         = load_night(edf_path)
        signals     = preprocess_signals(raw)
        mask, stats = remove_movement_artifacts(signals)

        save_dir = OUTPUT_DIR / ids["group"] / ids["subject_id"]
        save_dir.mkdir(parents=True, exist_ok=True)
        out_path = save_dir / (edf_path.stem + "_prep_raw.fif")
        raw.save(out_path, overwrite=True, verbose=False)

        log.update({
            "status":           "ok",
            "sfreq_original":   raw.info["sfreq"],
            "duration_hours":   round(raw.times[-1] / 3600, 3),
            "movement_pct_L":   stats.get(EEG_CH[0], ""),
            "movement_pct_R":   stats.get(EEG_CH[1], ""),
            "preprocessed_path": str(out_path),
        })

    except Exception as e:
        log["error"] = str(e)

    return log 

# ══════════════════════════════════════════════════════════════════════════════
# MAIN — batch verwerking
# ══════════════════════════════════════════════════════════════════════════════

def run_batch(limit: int = None):
    print("=" * 65)
    print("  EEG BATCH PREPROCESSING — Stap 3 t/m 7")
    print(f"  Start: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 65)

    print("\n[1/2] EDF-bestanden zoeken...")
    edf_files = find_edf_files(BASE_DIRS, limit=limit)

    if not edf_files:
        print("[STOP] Geen bestanden gevonden.")
        return

    print("\n[2/2] Preprocessing starten...\n")

    log_path = OUTPUT_DIR / "preprocessing_log.csv"
    log_rows = []
    n_ok, n_fail = 0, 0

    for i, edf_path in enumerate(edf_files, start=1):
        pct = i / len(edf_files) * 100
        print(f"  [{i:>4}/{len(edf_files)}  {pct:>5.1f}%]  {edf_path.name}", end="  →  ", flush=True)

        log = process_one_night(edf_path)
        log_rows.append(log)

        if log["status"] == "ok":
            n_ok += 1
            print(f"OK  ({log['duration_hours']} h  |  "
                  f"beweging L={log['movement_pct_L']}%  R={log['movement_pct_R']}%)")
        else:
            n_fail += 1
            print(f"FAIL  {log['error'][:80]}")

        # tussentijds opslaan elke 25 bestanden
        if i % 25 == 0 or i == len(edf_files):
            pd.DataFrame(log_rows).to_csv(log_path, index=False)

    print("\n" + "=" * 65)
    print(f"  Succesvol  : {n_ok}")
    print(f"  Mislukt    : {n_fail}")
    print(f"  Log        : {log_path}")
    print(f"  Einde      : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 65)

# ─── Start ────────────────────────────────────────────────────────────────────
if __name__ == "__main__":

    # !! EERST TESTEN op 3 bestanden !!
    # Als alles OK is: verander naar run_batch(limit=None)
    run_batch(limit=3)

NameError: name 'TARGET_SFREQ' is not defined

In [39]:
import matplotlib.pyplot as plt
from scipy import signal as scipy_signal

def verify_preprocessing(edf_path: Path):
    """
    Vergelijkt ruwe vs preprocessed data op 4 dingen:
    1. DC offset (mean ≈ 0 na removal)
    2. Notch filter (50 Hz piek weg)
    3. Bandpass (niets buiten 0.1–35 Hz)
    4. Resampling (sfreq = 128 Hz)
    """

    print(f"\n── Verificatie: {edf_path.name} ──")

    # ── Laad RAW (ongefilterd) ────────────────────────────────────
    raw_orig = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
    raw_orig.pick([ch for ch in EEG_CH if ch in raw_orig.ch_names])
    ch = raw_orig.ch_names[0]   # gebruik linker EEG kanaal
    data_raw = raw_orig.get_data(picks=ch)[0]
    sfreq_raw = raw_orig.info["sfreq"]

    # ── Laad PREPROCESSED (.fif) ──────────────────────────────────
    ids     = extract_ids(edf_path)
    fif_path = OUTPUT_DIR / ids["group"] / ids["subject_id"] / (edf_path.stem + "_prep_raw.fif")

    if not fif_path.exists():
        print(f"  [WARN] Geen .fif gevonden: {fif_path}")
        print(f"         Voer eerst run_batch() uit.")
        return

    raw_prep = mne.io.read_raw_fif(fif_path, preload=True, verbose=False)
    ch_prep  = raw_prep.ch_names[0]
    data_prep = raw_prep.get_data(picks=ch_prep)[0]
    sfreq_prep = raw_prep.info["sfreq"]

    # ══════════════════════════════════════════════════════════════
    # CHECK 1 — DC offset
    # ══════════════════════════════════════════════════════════════
    mean_raw  = data_raw.mean()
    mean_prep = data_prep.mean()
    dc_ok     = abs(mean_prep) < 1.0   # moet < 1 µV zijn

    print(f"\n  [1] DC offset:")
    print(f"      Voor  : {mean_raw:+.2f} µV")
    print(f"      Na    : {mean_prep:+.2f} µV  {'✓ OK' if dc_ok else '✗ PROBLEEM'}")

    # ══════════════════════════════════════════════════════════════
    # CHECK 2 — Resampling
    # ══════════════════════════════════════════════════════════════
    resamp_ok = sfreq_prep == TARGET_SFREQ
    print(f"\n  [2] Sampling frequency:")
    print(f"      Voor  : {sfreq_raw} Hz")
    print(f"      Na    : {sfreq_prep} Hz  {'✓ OK' if resamp_ok else '✗ PROBLEEM'}")

    # ══════════════════════════════════════════════════════════════
    # CHECK 3 + 4 — Power spectrum (notch + bandpass visueel)
    # ══════════════════════════════════════════════════════════════
    # Welch PSD voor beide signalen
    # nperseg=4*sfreq → frequentieresolutie van 0.25 Hz
    freqs_raw,  psd_raw  = scipy_signal.welch(data_raw,  fs=sfreq_raw,  nperseg=int(4*sfreq_raw))
    freqs_prep, psd_prep = scipy_signal.welch(data_prep, fs=sfreq_prep, nperseg=int(4*sfreq_prep))

    # Check 50 Hz power (notch)
    idx_50_raw  = np.argmin(np.abs(freqs_raw  - 50))
    idx_50_prep = np.argmin(np.abs(freqs_prep - 50))
    power_50_raw  = 10 * np.log10(psd_raw[idx_50_raw]  + 1e-30)
    power_50_prep = 10 * np.log10(psd_prep[idx_50_prep] + 1e-30)
    notch_ok = power_50_prep < power_50_raw - 10   # minstens 10 dB minder

    print(f"\n  [3] Notch filter (50 Hz power):")
    print(f"      Voor  : {power_50_raw:.1f} dB")
    print(f"      Na    : {power_50_prep:.1f} dB  {'✓ OK (>10 dB reductie)' if notch_ok else '✗ CHECK DIT'}")

    # Check bandpass: bijna geen power buiten 0.1–35 Hz
    mask_out  = freqs_prep > 36   # boven 35 Hz
    power_out = psd_prep[mask_out].mean() if mask_out.any() else 0
    mask_in   = (freqs_prep >= 1) & (freqs_prep <= 30)
    power_in  = psd_prep[mask_in].mean()
    bandpass_ok = power_out < power_in * 0.01   # buiten band < 1% van binnen

    print(f"\n  [4] Bandpass (0.1–35 Hz):")
    print(f"      Power in band (1–30 Hz) : {power_in:.4f}")
    print(f"      Power buiten (>36 Hz)   : {power_out:.4f}  {'✓ OK' if bandpass_ok else '✗ CHECK DIT'}")

    # ══════════════════════════════════════════════════════════════
    # PLOT — Tijdserie + Power spectrum
    # ══════════════════════════════════════════════════════════════
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    fig.suptitle(f"Preprocessing verificatie — {edf_path.stem}", fontsize=12)

    t_show = 30   # toon eerste 30 seconden

    # ── Tijdserie voor ───────────────────────────────────────────
    samp_raw  = int(t_show * sfreq_raw)
    t_raw     = np.arange(samp_raw) / sfreq_raw
    axes[0,0].plot(t_raw, data_raw[:samp_raw], lw=0.5, color="steelblue")
    axes[0,0].axhline(mean_raw, color="red", lw=1.5, linestyle="--", label=f"mean={mean_raw:+.1f} µV")
    axes[0,0].set_title("Tijdserie — VOOR preprocessing")
    axes[0,0].set_xlabel("Tijd (s)")
    axes[0,0].set_ylabel("Amplitude (µV)")
    axes[0,0].legend(fontsize=8)

    # ── Tijdserie na ────────────────────────────────────────────
    samp_prep = int(t_show * sfreq_prep)
    t_prep    = np.arange(samp_prep) / sfreq_prep
    axes[0,1].plot(t_prep, data_prep[:samp_prep], lw=0.5, color="darkorange")
    axes[0,1].axhline(mean_prep, color="red", lw=1.5, linestyle="--", label=f"mean={mean_prep:+.2f} µV")
    axes[0,1].set_title("Tijdserie — NA preprocessing")
    axes[0,1].set_xlabel("Tijd (s)")
    axes[0,1].set_ylabel("Amplitude (µV)")
    axes[0,1].legend(fontsize=8)

    # ── Power spectrum voor ──────────────────────────────────────
    axes[1,0].semilogy(freqs_raw, psd_raw, color="steelblue", lw=0.8)
    axes[1,0].axvline(50, color="red",  lw=1, linestyle="--", label="50 Hz")
    axes[1,0].axvline(0.1, color="green", lw=1, linestyle="--", label="0.1 Hz")
    axes[1,0].axvline(35, color="green", lw=1, linestyle="--", label="35 Hz")
    axes[1,0].set_title("Power spectrum — VOOR preprocessing")
    axes[1,0].set_xlabel("Frequentie (Hz)")
    axes[1,0].set_ylabel("Power (µV²/Hz)")
    axes[1,0].set_xlim(0, 70)
    axes[1,0].legend(fontsize=8)

    # ── Power spectrum na ────────────────────────────────────────
    axes[1,1].semilogy(freqs_prep, psd_prep, color="darkorange", lw=0.8)
    axes[1,1].axvline(50, color="red",   lw=1, linestyle="--", label="50 Hz (notch)")
    axes[1,1].axvline(0.1, color="green", lw=1, linestyle="--", label="0.1 Hz")
    axes[1,1].axvline(35,  color="green", lw=1, linestyle="--", label="35 Hz")
    axes[1,1].set_title("Power spectrum — NA preprocessing")
    axes[1,1].set_xlabel("Frequentie (Hz)")
    axes[1,1].set_ylabel("Power (µV²/Hz)")
    axes[1,1].set_xlim(0, 70)
    axes[1,1].legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"verify_{edf_path.stem}.png", dpi=120)
    plt.show()
    print(f"\n  Plot opgeslagen: verify_{edf_path.stem}.png")

    # ── Eindoordeel ───────────────────────────────────────────────
    all_ok = dc_ok and resamp_ok and notch_ok and bandpass_ok
    print(f"\n  {'✓ Preprocessing correct' if all_ok else '✗ Controleer bovenstaande punten'}")
    print("─" * 50)